# Lab 10: 3D Genome Structure - Introduction

This notebook contains solutions for all exercises covering Hi-C Contact Matrix Analysis:
- **Exercise 1**: Loading and displaying data
- **Exercise 2**: Compartment Calling
- **Exercise 3**: TAD Detection

---
## Exercise 1: Hi-C Contact Matrices

**Goals:**
1. Inspect a Hi-C Contact Matrix
2. Apply ICE or KR Normalization (Simplified)
3. PCA for A/B Compartment Detection

In [ ]:
#!uv add cooler
#!uv add h5py
import cooler
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def generate_toy_hic_matrix(N=100, decay=0.05, noise_level=0.1, tad_noise=(0.4, 0, 6), seed=None):
    """
    Generate a toy Hi-C contact matrix.
    
    Parameters:
        N (int): number of bins
        decay (float): exponential decay factor with genomic distance
        noise_level (float): standard deviation of Gaussian noise
        comp_noise (float): range of uniform intera-TAD noise
        seed (int): random seed for reproducibility
    
    Returns:
        mat (np.ndarray): N x N Hi-C contact matrix
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Distance-dependent decay
    mat = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            mat[i, j] = np.exp(-decay * abs(i-j))
    
    # TAD-like domains
    for start in range(0, N, N//5):
        end = start + N//5
        mat[start:end, start:end] += np.random.uniform(
            low=tad_noise[0], high=tad_noise[1], size=(end - start, end - start)
        )
    
    # Add Gaussian noise
    mat += np.random.normal(0, noise_level, size=(N,N))
    mat[mat<0] = 0  # no negative contacts
    
    return mat
def plot_hic_matrix(mat, title="Hi-C Contact Map", cmap='Reds', max_color=None):
    """Plot heatmap of Hi-C matrix"""
    plt.figure(figsize=(8, 8))
    if isinstance(max_color, float):
        max_color = np.percentile(mat, max_color)    
    plt.imshow(mat, origin='upper', cmap=cmap, vmax=max_color)
    plt.colorbar(label="Contact strength")
    plt.title(title)
    plt.xlabel("Bin")
    plt.ylabel("Bin")
    plt.tight_layout()
    plt.show()

In [ ]:
# Generate synthetic Hi-C matrix
toy_hic = generate_toy_hic_matrix(
    N=100,
    noise_level=0.05,
    seed=42
)

print(f"Hi-C matrix shape: {toy_hic.shape}")
print(f"Min value: {toy_hic.min():.4f}, Max value: {toy_hic.max():.4f}")

plot_hic_matrix(toy_hic, title="Synthetic Toy Hi-C Contact Map")

### ICE Normalization (Simplified)

Since reading the `.hic` (the `hic-straw` library) format is sometimes problematic, we can use an alternative: `.mcool`, which is just a HDF5 file with a specific structure. You can obtain it from a `.hic` file using `hic2cool` package.

In [ ]:
def ice_normalization(mat, max_iter=100, tolerance=1e-5):
    """
    Simplified ICE (Iterative Correction and Eigenvector decomposition) normalization.
    """
    mat = mat.copy().astype(float)
    n = mat.shape[0]
    
    # Replace zeros with small value to avoid division issues
    mat[mat == 0] = 1e-10
    
    bias = np.ones(n)
    
    for iteration in range(max_iter):
        # Compute row sums
        row_sums = mat.sum(axis=1)
        
        # Avoid division by zero
        row_sums[row_sums == 0] = 1
        
        # Update bias
        target_sum = np.mean(row_sums)
        correction = target_sum / row_sums
        
        # Apply correction
        mat = mat * np.outer(correction, correction)
        bias *= correction
        
        # Check convergence
        if np.max(np.abs(correction - 1)) < tolerance:
            print(f"ICE converged after {iteration + 1} iterations")
            break
    
    return mat, bias


# Apply ICE normalization
ice_normalized, bias = ice_normalization(toy_hic)
plot_hic_matrix(ice_normalized, title="ICE Normalized Hi-C")

In [ ]:
def scan_hdf5_file(path):
    with h5py.File(path) as f:
        def show(name, obj):
            if isinstance(obj, h5py.Group):
                print(f"{name} (Group)")
            else:
                print(f"{name} (Dataset) shape={obj.shape} dtype={obj.dtype}")
        f.visititems(show)

_mcool_file = './lab_outputs/rao/GSE63525_GM12878_insitu_primary+replicate_combined.mcool'
scan_hdf5_file(_mcool_file)

In [ ]:
def fetch_region_cooler(mcool_path, chrom, start, end, resolution):
    if chrom.startswith('chr'):
        chrom = chrom[3:]  # remove 'chr' prefix for cooler
    c = cooler.Cooler(f"{mcool_path}::resolutions/{resolution}")
    return c.matrix(balance=False).fetch(f"{chrom}:{start}-{end}")

sample_region_small = ('chr2', 134_000_000, 138_000_000)
sample_region_small_hic = fetch_region_cooler(_mcool_file, *sample_region_small, 25_000)
plot_hic_matrix(sample_region_small_hic, title="Hi-C Contact Map", max_color=99.0)

---
## Exercise 2: PCA for A/B Compartment Detection

**A/B Compartments** are large-scale chromatin organization patterns:
- **Compartment A**: Gene-rich, active, open chromatin
- **Compartment B**: Gene-poor, inactive, closed chromatin

In Hi-C data, compartments create a **checkerboard pattern** where same-type regions have higher contact frequencies. We detect them using PCA on the O/E (observed/expected) correlation matrix.

First, let's generate a Hi-C matrix with clear compartment structure:

In [ ]:
from sklearn.decomposition import PCA

def generate_compartment_hic(N=100, n_compartments=6, compartment_strength=0.5, 
                              decay=0.02, noise_level=0.02, seed=42):
    """
    Generate a Hi-C matrix with clear A/B compartment structure.
    
    Compartments create a checkerboard pattern where:
    - Same-type regions (A-A or B-B) have higher contact frequencies
    - Different-type regions (A-B) have lower contact frequencies
    """
    np.random.seed(seed)
    
    # Create alternating compartment labels: A=1, B=-1
    compartment_size = N // n_compartments
    compartment_labels = np.zeros(N)
    for i in range(N):
        region = i // compartment_size
        compartment_labels[i] = 1 if region % 2 == 0 else -1
    
    # Build the contact matrix
    mat = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            # Distance-dependent decay (polymer physics)
            distance_term = np.exp(-decay * abs(i - j))
            
            # Compartment term: same compartment = boost, different = reduce
            # This creates the checkerboard pattern
            same_compartment = compartment_labels[i] * compartment_labels[j]
            compartment_term = 1 + same_compartment * compartment_strength
            
            mat[i, j] = distance_term * compartment_term
    
    # Add noise and ensure symmetry
    noise = np.random.normal(0, noise_level, size=(N, N))
    noise = (noise + noise.T) / 2
    mat = mat + noise
    mat[mat < 0] = 0
    
    return mat, compartment_labels


# Generate Hi-C with compartment structure
compartment_hic, true_labels = generate_compartment_hic(
    N=100, 
    n_compartments=6,  # Creates 3 A and 3 B regions
    compartment_strength=0.6,
    decay=0.015,
    noise_level=0.02
)

print(f"Generated Hi-C matrix with {6} alternating compartment regions")
print(f"True labels: {int(np.sum(true_labels > 0))} bins in A, {int(np.sum(true_labels < 0))} bins in B")

# Visualize - should show checkerboard pattern
plot_hic_matrix(compartment_hic, title="Hi-C Matrix with A/B Compartments (Checkerboard Pattern)")

In [ ]:
def detect_compartments(mat):
    """
    Detect A/B compartments using PCA on O/E correlation matrix.
    
    Steps:
    1. Compute expected contacts at each genomic distance
    2. Calculate O/E (observed/expected) matrix - removes distance decay
    3. Compute correlation matrix of O/E rows
    4. PC1 of correlation matrix separates A and B compartments
    """
    n = mat.shape[0]
    
    # Compute expected contacts at each distance
    expected = np.zeros_like(mat)
    for i in range(n):
        for j in range(n):
            expected[i, j] = 1 # mean of pixels at distance d=|i - j| form diagonal (or "1")
    
    # Guard against zero div.
    expected[expected == 0] = 1e-10
    
    # O/E (observed vs expected) matrix removes the distance decay effect
    # Use log2 for better distribution
    oe_mat = # TODO: compute O/E matrix (log2)

    # Fix the matrix again:
    oe_mat = np.nan_to_num(oe_mat)
    
    # Compute correlation matrix: bins in the same compartment will be correlated
    corr_mat = np.corrcoef(oe_mat)
    corr_mat = np.nan_to_num(corr_mat)
    
    # PCA: PC1 captures the compartment signal
    pca = None # TODO PCA
    pc1 = None # COmponent 1
    
    # Assign compartments based on PC1 sign
    compartments = np.where(pc1 > 0, 'A', 'B')
    
    return pc1, compartments, corr_mat, oe_mat


# Detect compartments using PCA
pc1, compartments, corr_mat, oe_mat = detect_compartments(compartment_hic)

print(f"Detected compartments: A={np.sum(compartments=='A')}, B={np.sum(compartments=='B')}")
# Comprehensive visualization of compartment detection
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Row 1: Hi-C, O/E, and Correlation matrices
ax1 = axes[0, 0]
im1 = ax1.imshow(compartment_hic, cmap='Reds', origin='upper')
ax1.set_title('Original Hi-C Matrix')
ax1.set_xlabel('Bin')
ax1.set_ylabel('Bin')
plt.colorbar(im1, ax=ax1, shrink=0.8)

ax2 = axes[0, 1]
im2 = ax2.imshow(oe_mat, cmap='RdBu_r', origin='upper', vmin=-2, vmax=2)
ax2.set_title('O/E Matrix (log2)')
ax2.set_xlabel('Bin')
plt.colorbar(im2, ax=ax2, shrink=0.8)

ax3 = axes[0, 2]
im3 = ax3.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1, origin='upper')
ax3.set_title('Correlation Matrix\n(Checkerboard = Compartments)')
ax3.set_xlabel('Bin')
plt.colorbar(im3, ax=ax3, shrink=0.8)

# Row 2: PC1 signal, compartment track, and comparison with ground truth
ax4 = axes[1, 0]
ax4.plot(pc1, 'k-', linewidth=1)
ax4.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax4.fill_between(range(len(pc1)), pc1, where=pc1 > 0, alpha=0.5, color='red', label='A')
ax4.fill_between(range(len(pc1)), pc1, where=pc1 <= 0, alpha=0.5, color='blue', label='B')
ax4.set_xlabel('Bin')
ax4.set_ylabel('PC1 Value')
ax4.set_title('PC1 Signal (Compartment Eigenvector)')
ax4.legend(loc='upper right')

ax5 = axes[1, 1]
# Predicted compartments
pred_colors = [1 if c == 'A' else -1 for c in compartments]
ax5.imshow([pred_colors], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax5.set_title('Predicted Compartments')
ax5.set_xlabel('Bin')
ax5.set_yticks([0])
ax5.set_yticklabels(['Predicted'])

ax6 = axes[1, 2]
# Compare with ground truth
ax6.imshow([true_labels], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax6.set_title('Ground Truth Compartments')
ax6.set_xlabel('Bin')
ax6.set_yticks([0])
ax6.set_yticklabels(['True'])

plt.tight_layout()
plt.show()

# Calculate accuracy
# Note: PC1 sign might be flipped, so check both orientations
pred_binary = np.array([1 if c == 'A' else -1 for c in compartments])
accuracy1 = np.mean(pred_binary == true_labels)
accuracy2 = np.mean(pred_binary == -true_labels)  # Check if sign is flipped
accuracy = max(accuracy1, accuracy2)

print(f"\nCompartment detection accuracy: {accuracy:.1%}")
print(f"(Note: PC1 sign is arbitrary, so we check both orientations)")

In [ ]:
sample_region_big = ('chr2', 120_000_000, 160_000_000)
sample_compartment_resolution = 100_000
sample_region_big_hic_mat = fetch_region_cooler(_mcool_file, *sample_region_big, sample_compartment_resolution)

plot_hic_matrix(sample_region_big_hic_mat, title="Hi-C Contact Map for Larger Region", max_color=99.0)

In [ ]:
def read_compartments_from_bed(bed_path, chrom, start, end, resolution):
    """
    Read compartment annotations from a BED file and convert to bin labels.
    
    Parameters:
        bed_path (str): path to BED file with compartments
        chrom (str): chromosome name
        start (int): start position
        end (int): end position

    Returns:
        np.ndarray: array of compartment labels for each bin
    """
    df = pd.read_csv(bed_path, sep='\t', header=None, usecols=[0, 1, 2, 3], names=['chrom', 'start', 'end', 'compartment'])
    df = df[(df['chrom'] == chrom) & (df['end'] > start) & (df['start'] < end)]
    df['compartment'] = df['compartment'].str[0]  # Extract A/B from subcompartment (e.g., A1 -> A)
    bin_offset = start // resolution
    bin_compartments = np.full((end - start) // resolution, 'B', dtype='<U1')  # Default to B
    for _, row in df.iterrows():
        bin_start = max(0, row['start'] // resolution - bin_offset)
        bin_end = min(len(bin_compartments), (row['end'] // resolution) - bin_offset)
        if row['compartment'] == 'A':
            bin_compartments[bin_start:bin_end] = 'A'
    return bin_compartments
    
rao_compartment_labels = read_compartments_from_bed(
    './lab_outputs/rao/GSE63525_GM12878_subcompartments.bed',
    *sample_region_big, sample_compartment_resolution
)

---
## Exercise 3: TAD Detection

**Goals:**
1. Detect TAD boundaries using insulation score

In [ ]:
def compute_insulation_score(mat, window=5):
    """Compute insulation score for TAD boundary detection."""
    n = mat.shape[0]
    insulation = np.zeros(n)
    for i in range(n):
        start = max(0, i-window)
        end = min(n, i+window+1)
        submat = mat[start:end, start:end]
        insulation[i] = submat.sum()
    insulation = -insulation  # Lower values = boundaries
    return insulation


def detect_tad_boundaries(insulation):
    """Detect TAD boundaries as local minima in insulation score."""
    n = len(insulation)
    boundaries = [
        i for i in range(1, n-1)
        if insulation[i] < insulation[i-1] and insulation[i] < insulation[i+1]
    ]
    return boundaries


def get_tad_regions(boundaries, n_bins):
    """Convert TAD boundaries to a list of TAD regions (start, end)."""
    regions = []
    start = 0
    for b in boundaries:
        regions.append((start, b))
        start = b
    regions.append((start, n_bins))
    return regions